<a href="https://colab.research.google.com/github/ASR181/Contract-Audit-pipeline/blob/main/contract_audit_pipeline_capstone_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Team Members:

*   Abdullah Abdulmohsen Abdullah Alotaibi
*   Saud Abdulkarim Saud Almotawa
*   Nawaf Faisal Saad Aldhafyan

# Automated Contract Audit & Compliance Pipeline
This notebook implements the Multi-Agent Workflow using LangGraph, ChromaDB for Active Knowledge Retrieval, Tavily for external search, PyPDF2 for document extraction, ChatGroq for LLM processing, and an immutable SQLite audit trail tracking latency.

In [ ]:
# Cell 1: Install required dependencies
!pip install -qU langchain langgraph langchain-groq langchain-huggingface chromadb arize-phoenix langchain-community chromadb sentence-transformers openinference-instrumentation-langchain PyPDF2 reportlab pandas tavily-python
print("installing packages done!")

installing packages done!


In [ ]:
# Cell 2: Imports and Observability Setup (Arize Phoenix)
import os
import json
import operator
import time
import sqlite3
import pandas as pd
from typing import TypedDict, Annotated, Sequence
from datetime import datetime, timezone, timedelta
import uuid
import PyPDF2
import re
from IPython.display import display

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from langchain_community.tools.tavily_search import TavilySearchResults

from google.colab import userdata, files
from google.colab.output import eval_js
import phoenix as px

from openinference.instrumentation.langchain import LangChainInstrumentor

# Set API Keys
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Add it in the Colab Secrets panel (🔑).")

if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY not found. Add it in the Colab Secrets panel (🔑).")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

# Launch Phoenix Tracing

# 1. Launch Phoenix
session = px.launch_app()

# 2. Force the URL generation using Colab's proxy
# Phoenix runs on port 6006 by default
phoenix_url = eval_js("google.colab.kernel.proxyPort(6006)")

# 3. Instrument LangChain
from openinference.instrumentation.langchain import LangChainInstrumentor
LangChainInstrumentor().instrument()

print("Imports complete.")
print(f"Phoenix tracing is running at: {phoenix_url}")

🌍 To view the Phoenix app in your browser, visit https://082cg14wezh6-496ff2e9c6d22116-6006-colab.googleusercontent.com/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


Imports complete.
Phoenix tracing is running at: https://6006-m-s-kkb-use1d0-qdhtsdjji1ri-d.us-east1-0.prod.colab.dev


In [ ]:
# Cell 3: Setup ChromaDB for Active Knowledge Retrieval
print("Initializing local embeddings model (HuggingFace)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

texts = [

    # ===========================
    # Payment Policies
    # ===========================
    "Vendor invoices must be paid within 30 calendar days.",
    "Any payment term exceeding 30 days requires CFO approval.",

    # ===========================
    # Confidentiality
    # ===========================
    "Every vendor contract must contain a confidentiality clause.",
    "Confidential information cannot be disclosed to third parties without written authorization.",

    # ===========================
    # Data Privacy
    # ===========================
    "Customer personal information must never be shared with third parties without consent.",
    "Vendors processing personal information must comply with GDPR or equivalent privacy regulations.",

    # ===========================
    # Encryption
    # ===========================
    "Sensitive customer information must be encrypted using AES-256 or stronger encryption.",
    "Data must remain encrypted during storage and transmission.",

    # ===========================
    # Vendor Security
    # ===========================
    "Vendors must maintain ISO 27001 or equivalent security certification.",
    "Critical vendors must perform annual penetration testing.",

    # ===========================
    # Contract Duration
    # ===========================
    "Contracts exceeding three years require executive approval.",
    "Long-term agreements must undergo annual compliance review.",

    # ===========================
    # Liability
    # ===========================
    "Liability cannot be capped below the total contract value.",
    "Vendors remain liable for security breaches caused by negligence.",

    # ===========================
    # Data Retention
    # ===========================
    "Customer data must not be retained for more than five years unless legally required.",
    "Expired customer information must be securely deleted.",

    # ===========================
    # Termination
    # ===========================
    "Termination notice periods must not exceed 90 days without executive approval.",
    "The company reserves the right to terminate contracts immediately following severe compliance violations."
]

metadatas = [

    {"category": "Payment"},
    {"category": "Payment"},

    {"category": "Confidentiality"},
    {"category": "Confidentiality"},

    {"category": "Privacy"},
    {"category": "Privacy"},

    {"category": "Encryption"},
    {"category": "Encryption"},

    {"category": "Security"},
    {"category": "Security"},

    {"category": "Contract Duration"},
    {"category": "Contract Duration"},

    {"category": "Liability"},
    {"category": "Liability"},

    {"category": "Retention"},
    {"category": "Retention"},

    {"category": "Termination"},
    {"category": "Termination"},
]

vectorstore = Chroma(
    collection_name="enterprise_research_memory",
    embedding_function=embeddings,
    persist_directory="./enterprise_memory_db"
)

vectorstore.add_texts(texts=texts, metadatas=metadatas)
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3
    }
)
print(f"Knowledge Base contains {len(texts)} company compliance policies.")
print("Persistent ChromaDB Vector Store populated with company policies.")

Initializing local embeddings model (HuggingFace)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge Base contains 18 company compliance policies.
Persistent ChromaDB Vector Store populated with company policies.


In [ ]:
# Cell 4: Document Processing & Simulated Cloud Storage

print("=" * 60)
print("Vendor Contract Upload & Cloud Storage")
print("=" * 60)

# 1. Simulate MinIO/S3 Cloud Storage Bucket
class SimulatedMinIOBucket:
    def __init__(self, bucket_name):
        self.bucket_name = bucket_name
        self.storage_path = f"/tmp/simulated_s3/{bucket_name}"
        os.makedirs(self.storage_path, exist_ok=True)
        print(f"[Cloud Storage] Initialized simulated bucket: s3://{self.bucket_name}")

    def upload_file(self, filename, file_data):
        file_path = os.path.join(self.storage_path, filename)
        with open(file_path, "wb") as f:
            f.write(file_data)
        print(f"[Cloud Storage] File '{filename}' securely uploaded to s3://{self.bucket_name}")
        return file_path

# Initialize our mock cloud environment
cloud_bucket = SimulatedMinIOBucket("vendor-contracts-secure")

# 2. Upload via Colab wrapped in Try-Except
try:
    uploaded = files.upload()

    # Check if a file was actually selected
    if not uploaded:
        raise ValueError("No file was selected for upload.")

    pdf_name = next(iter(uploaded.keys()))

    # Check 1: File Extension Validation
    if not pdf_name.lower().endswith('.pdf'):
        raise ValueError(f"Invalid file type. Expected a PDF, but got: '{pdf_name}'")

    # 3. Store in Simulated Cloud Bucket
    contract_path = cloud_bucket.upload_file(pdf_name, uploaded[pdf_name])

    # 4. Extract Text
    metadata = {
        "contract_id": str(uuid.uuid4()),
        "filename": pdf_name,
        "upload_time": datetime.now().isoformat(),
        "file_size_kb": round(os.path.getsize(contract_path) / 1024, 2)
    }

    # Check 2: PyPDF2 Integrity Validation
    try:
        reader = PyPDF2.PdfReader(contract_path)
        metadata["pages"] = len(reader.pages)
        contract_text = ""

        for page in reader.pages:
            text = page.extract_text()
            if text:
                contract_text += text + "\n"

    except PyPDF2.errors.PdfReadError:
        raise ValueError(f"The file '{pdf_name}' is corrupted or not a valid PDF document.")

    # Clean and split into clauses
    clauses = [
        c.strip()
        for c in re.split(r"\.\s+|\n+", contract_text)
        if len(c.strip()) > 20
    ]

    contract_document = {
        "metadata": metadata,
        "raw_text": contract_text,
        "clauses": clauses
    }

    print("\n" + "=" * 60)
    print("Contract Summary")
    print("=" * 60)
    print(f"Filename: {metadata['filename']}")
    print(f"Pages: {metadata['pages']}")
    print(f"Clauses: {len(clauses)}")

except Exception as e:
    print("\n❌ [ERROR] File Processing Failed:")
    print(str(e))
    print("Please restart this cell and upload a valid PDF document.")

    contract_document = None

    # forces the notebook cell to fail and halt execution
    raise e

Vendor Contract Upload & Cloud Storage
[Cloud Storage] Initialized simulated bucket: s3://vendor-contracts-secure


Saving contract 2233.pdf to contract 2233 (1).pdf
[Cloud Storage] File 'contract 2233 (1).pdf' securely uploaded to s3://vendor-contracts-secure

Contract Summary
Filename: contract 2233 (1).pdf
Pages: 3
Clauses: 117


In [ ]:
# Cell 5: Simulated Secure Database (Audit Persistence)

# 1. Initialize Simulated Secure Database for Immutable Audit Trail
db_path = "secure_audit_trail.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# FIX: Drop the old table to force the new schema to apply
cursor.execute('DROP TABLE IF EXISTS audit_log')

# 2. Create Audit Table ensuring strict data isolation
cursor.execute('''
    CREATE TABLE audit_log (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT,
        batch_id TEXT,
        latency_seconds REAL,
        total_tokens INTEGER,
        estimated_cost_usd REAL,
        compatibility_score REAL,
        critical_errors INTEGER,
        findings TEXT
    )
''')
conn.commit()

# 3. Updated Insert Function to include Score and Errors
def log_audit_record(batch_id, latency, tokens, cost, comp_score, errors, findings):
    cursor.execute('''
        INSERT INTO audit_log (
            timestamp, batch_id, latency_seconds, total_tokens,
            estimated_cost_usd, compatibility_score, critical_errors, findings
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        datetime.now(timezone(timedelta(hours=3))).isoformat(),
        batch_id, latency, tokens, cost, comp_score, errors, findings
    ))
    conn.commit()

print("Secure Immutable Audit Database Initialized (Old schema wiped).")

Secure Immutable Audit Database Initialized (Old schema wiped).


In [ ]:
# ======================================================
# Cell 6: Define Agent State for LangGraph
# ======================================================

class AgentState(TypedDict):
    """
    Shared state passed between LangGraph nodes.
    """

    # Conversation history
    messages: Annotated[Sequence[BaseMessage], operator.add]

    # Next node selected by the Supervisor
    next_node: str

    # Extracted contract text
    contract_text: str

    # Structured compliance findings
    compliance_issues: list[dict]

    # Final report generated by Reporting Agent
    audit_report: str


# ======================================================
# LLM Configuration
# ======================================================

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# ======================================================
# External Search Tool
# ======================================================

tavily_tool = TavilySearchResults(
    max_results=1
)

print("LangGraph AgentState defined.")
print("ChatGroq model initialized.")
print("Tavily Search Tool initialized.")

LangGraph AgentState defined.
ChatGroq model initialized.
Tavily Search Tool initialized.


In [ ]:
# Cell 7: Compliance Agent (Worker) - Optimized for API Limits

class ComplianceAgent:
    def __init__(self, llm, retriever, search_tool):
        self.llm = llm
        self.retriever = retriever
        self.search_tool = search_tool

    def __call__(self, state: AgentState):
        print(">> [Node: Compliance Agent] Triaging and batching contract clauses...")
        contract_text = state.get("contract_text", "")

        risk_keywords = ["liability", "terminate", "payment", "confidential", "data", "privacy", "gdpr", "pii", "breach", "security"]
        raw_clauses = [c.strip() for c in contract_text.split('\n') if len(c.strip()) > 30]

        target_clauses = [c for c in raw_clauses if any(keyword in c.lower() for keyword in risk_keywords)]
        print(f"   - Triaged {len(raw_clauses)} total clauses down to {len(target_clauses)} high-risk clauses.")

        batch_size = 5
        batches = [target_clauses[i:i + batch_size] for i in range(0, len(target_clauses), batch_size)]

        all_issues = []

        for i, batch in enumerate(batches):
            start_time = time.time()
            batch_id = f"Batch {i+1}"
            print(f"\n>> Processing {batch_id}...")

            batch_text = "\n".join([f"[{idx+1}] {clause}" for idx, clause in enumerate(batch)])
            docs = self.retriever.invoke(f"Policies regarding: {batch[0]}")
            policies = "\n".join([d.page_content for d in docs[:3]])

            external_context = ""
            if "PII" in batch_text or "GDPR" in batch_text:
                print("   [Tool Call] Fetching external GDPR context...")
                search_res = self.search_tool.invoke("GDPR vendor data processing requirements")
                external_context = f"\nExternal Regulatory Context:\n{search_res}"

            # FORCE STRUCTURED JSON OUTPUT
            prompt = ChatPromptTemplate.from_messages([
                ("system", """You are a legal audit assistant. Review the batch of contract clauses against internal policies.
                You MUST respond with ONLY a valid JSON object. Do not use markdown blocks (```). Do not add any conversational text.
                The JSON must contain exactly three keys:
                "compatibility_score": (float) A score from 0.0 to 100.0 representing overall compliance.
                "critical_errors": (integer) Total number of severe policy violations found.
                "findings": (string) A concise summary of the risks.

                Internal Policies:
                {policies}{external_context}
                """),
                ("user", "Evaluate these clauses:\n{batch_text}")
            ])

            chain = prompt | self.llm

            try:
                response = chain.invoke({"policies": policies, "external_context": external_context, "batch_text": batch_text})

                # ROBUST JSON PARSING
                try:
                    # Find the content between the first { and the last }
                    match = re.search(r'\{[\s\S]*\}', response.content)
                    if match:
                        clean_json = match.group(0)
                        parsed_data = json.loads(clean_json)
                    else:
                        raise ValueError("No JSON bracket found in response.")

                    comp_score = float(parsed_data.get("compatibility_score", 0.0))
                    errors = int(parsed_data.get("critical_errors", 0))
                    findings = parsed_data.get("findings", "No findings provided.")

                except Exception as parse_err:
                    print(f"   ⚠️ [WARNING] Failed to parse LLM JSON ({parse_err}). Defaulting metrics.")
                    comp_score = 0.0
                    errors = -1
                    findings = response.content

            except Exception as e:
                print(f"   ⚠️ [API ERROR] {e}")
                time.sleep(10)
                continue

            latency = time.time() - start_time

            # Extract Tokens and Calculate Cost
            tokens = 0
            if hasattr(response, 'usage_metadata') and response.usage_metadata:
                tokens = response.usage_metadata.get('total_tokens', 0)

            cost = (tokens / 1_000_000) * 0.05

            # Log to DB
            log_audit_record(batch_id, latency, tokens, cost, comp_score, errors, findings)

            # Append to LangGraph state
            all_issues.append(f"--- {batch_id} (Score: {comp_score}%, Errors: {errors}) ---\n{findings}")

            print(f"   - {batch_id} complete | Score: {comp_score}% | Errors: {errors} | Cost: ${cost:.6f}")
            time.sleep(2)

        print("\nCompliance evaluation complete.")
        return {
            "messages": [AIMessage(content="Compliance evaluation complete.")],
            "compliance_issues": all_issues
        }

compliance_agent_node = ComplianceAgent(llm, retriever, tavily_tool)
print("Compliance Agent updated for Bulletproof JSON Extraction.")

Compliance Agent updated for Bulletproof JSON Extraction.


In [ ]:
# Cell 8: Reporting Agent (Worker)

class ReportingAgent:
    def __init__(self, llm):
        self.llm = llm

    def __call__(self, state: AgentState):
        print(">> [Node: Reporting Agent] Generating final audit summary...")
        issues = state.get("compliance_issues", [])
        issues_text = "\n\n".join(issues)

        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a Reporting Agent. Create a final audit summary and recommendation based on the compliance issues found."),
            ("user", "Compliance Issues:\n{issues}")
        ])

        chain = prompt | self.llm
        response = chain.invoke({"issues": issues_text})

        print("Report generation complete.")
        return {
            "messages": [AIMessage(content=f"Final Report: {response.content}")],
            "audit_report": response.content
        }

reporting_agent_node = ReportingAgent(llm)
print("Reporting Agent node defined and instantiated.")

Reporting Agent node defined and instantiated.


In [ ]:
# Cell 9: Supervisor Orchestrator

class SupervisorAgent:
    def __call__(self, state: AgentState):
        print(">> [Node: Supervisor] Routing workflow based on LangGraph state...")
        if not state.get("compliance_issues"):
            print("   -> Routing to compliance_agent")
            return {"next_node": "compliance_agent"}
        elif not state.get("audit_report"):
            print("   -> Routing to reporting_agent")
            return {"next_node": "reporting_agent"}
        else:
            print("   -> Routing to END")
            return {"next_node": END}

supervisor_node = SupervisorAgent()
print("Supervisor Orchestrator node defined and instantiated.")

Supervisor Orchestrator node defined and instantiated.


In [ ]:
# Cell 10: Build and Compile the LangGraph

def dynamic_router(state: AgentState):
    return state.get("next_node")

def audit_node(state: AgentState):
    print(">> [Node: Audit] Finalizing audit workflow...")
    return {"messages": [AIMessage(content="Audit workflow finalized.")]}

workflow = StateGraph(AgentState)

workflow.add_node("supervisor", supervisor_node)
workflow.add_node("compliance_agent", compliance_agent_node)
workflow.add_node("reporting_agent", reporting_agent_node)
workflow.add_node("audit", audit_node)

workflow.set_entry_point("supervisor")

# Define conditional edges from supervisor
workflow.add_conditional_edges(
    "supervisor",
    dynamic_router,
    {
        "compliance_agent": "compliance_agent",
        "reporting_agent": "reporting_agent",
        END: END
    }
)

# Workers and Final Audit edges
workflow.add_edge("compliance_agent", "supervisor")
workflow.add_edge("reporting_agent", "audit")
workflow.add_edge("audit", END)

app = workflow.compile()
print("LangGraph built and compiled successfully.")

LangGraph built and compiled successfully.


In [ ]:
# Cell 11: Execute Pipeline using PyPDF2 Extracted Text

print("Starting LangGraph Audit Pipeline with Extracted PDF Text")
initial_state = {
    "messages": [HumanMessage(content="Please audit this contract.")],
    "contract_text": contract_document["raw_text"],
    "next_node": "compliance_agent",
    "compliance_issues": [],
    "audit_report": ""
}

final_value = None
for output in app.stream(initial_state):
    for key, value in output.items():
        final_value = value

print("\n--- FINAL AUDIT REPORT ---")
if final_value:
    print(final_value.get("audit_report", "Report complete."))

print("Pipeline execution complete. View Phoenix Traces at: http://localhost:6006")

Starting LangGraph Audit Pipeline with Extracted PDF Text
>> [Node: Supervisor] Routing workflow based on LangGraph state...
   -> Routing to compliance_agent
>> [Node: Compliance Agent] Triaging and batching contract clauses...
   - Triaged 101 total clauses down to 11 high-risk clauses.

>> Processing Batch 1...
   - Batch 1 complete | Score: 20.0% | Errors: 0 | Cost: $0.000018

>> Processing Batch 2...
   - Batch 2 complete | Score: 60.0% | Errors: 2 | Cost: $0.000017

>> Processing Batch 3...
   - Batch 3 complete | Score: 0.0% | Errors: 1 | Cost: $0.000012

Compliance evaluation complete.
>> [Node: Supervisor] Routing workflow based on LangGraph state...
   -> Routing to reporting_agent
>> [Node: Reporting Agent] Generating final audit summary...
Report generation complete.
>> [Node: Audit] Finalizing audit workflow...

--- FINAL AUDIT REPORT ---
Report complete.
Pipeline execution complete. View Phoenix Traces at: http://localhost:6006


In [ ]:
# Cell 12: View Immutable Compliance Audit Trail

print("🔒 --- SECURE IMMUTABLE AUDIT TRAIL ---")

df_audit = pd.read_sql_query("""
    SELECT
        timestamp,
        batch_id,
        compatibility_score AS "Score (%)",
        critical_errors AS "Violations",
        ROUND(latency_seconds, 2) AS "Latency (s)",
        ROUND(estimated_cost_usd, 6) AS "Cost ($)"
    FROM audit_log
""", conn)

display(df_audit)

🔒 --- SECURE IMMUTABLE AUDIT TRAIL ---


,timestamp,batch_id,Score (%),Violations,Latency (s),Cost ($)
0,2026-06-26T18:21:10.358609+03:00,Batch 1,20.0,0,0.32,0.000018
1,2026-06-26T18:21:12.620349+03:00,Batch 2,60.0,2,0.25,0.000017
2,2026-06-26T18:21:15.062084+03:00,Batch 3,0.0,1,0.42,0.000012
